# DIC Comparison Table

Collects DIC from all model outputs (pyjags + R), adds descriptions, and saves:
-  — read by thesis Rmd
-  — exploratory reference

Run this after fitting any new model.

In [7]:
import importlib
import sys
from pathlib import Path

import pandas as pd

# Works whether the kernel cwd is CODE/ or CODE/pyjags_pipeline/
_here = Path().resolve()
PROJECT_ROOT = _here.parent if _here.name == 'pyjags_pipeline' else _here
PIPELINE_ROOT = PROJECT_ROOT / 'pyjags_pipeline'
OUTPUT_ROOT   = PIPELINE_ROOT / 'output'
R_OUTPUT_ROOT = PROJECT_ROOT / 'data' / 'models'

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root : {PROJECT_ROOT}')
print(f'Pipeline root: {PIPELINE_ROOT}')

Project root : /Users/fluffypony/Library/Mobile Documents/com~apple~CloudDocs/Documents/Galway/PROJECT/CODE
Pipeline root: /Users/fluffypony/Library/Mobile Documents/com~apple~CloudDocs/Documents/Galway/PROJECT/CODE/pyjags_pipeline


In [8]:
# --- Collect DIC from pyjags outputs ---
# Only include models that have a registered def file in pyjags_pipeline/defs/
# This prevents orphaned output directories (e.g. exploratory runs) from
# appearing in the thesis table.

def _load_dic(version):
    path = OUTPUT_ROOT / version / 'dic.csv'
    if path.exists():
        df = pd.read_csv(path)
        return {
            'version': version,
            'deviance': float(df['deviance'].iloc[0]),
            'penalty':  float(df['penalty'].iloc[0]),
            'DIC':      float(df['DIC'].iloc[0]),
        }
    return None


# Derive versions from registered defs (vX_Y.py -> vX.Y), not from output dirs
def _def_versions():
    versions = []
    for path in sorted((PIPELINE_ROOT / 'defs').glob('v*.py')):
        if path.stem == '__init__':
            continue
        # convert filename v6_1 -> v6.1
        versions.append(path.stem.replace('_', '.', 1))
    return versions


versions = _def_versions()

rows = [_load_dic(v) for v in versions]
df = pd.DataFrame([r for r in rows if r]).sort_values('DIC').reset_index(drop=True)
print(f'Found {len(df)} pyjags models')
df

KeyError: 'DIC'

In [ ]:
# --- Load human-readable names from model defs ---

def _load_model_names():
    names = {}
    for path in sorted((PIPELINE_ROOT / 'defs').glob('v*.py')):
        try:
            mod = importlib.import_module(f'pyjags_pipeline.defs.{path.stem}')
            m = mod.Model()
            names[m.version.lower()] = m.name  # key: 'v6.1', not 'v6_1'
        except Exception as e:
            print(f'  warning: {path.stem}: {e}')
    return names


names = _load_model_names()
print(f'Loaded {len(names)} model names')

df['description'] = df['version'].str.lower().map(names).fillna('')
df

In [ ]:
# --- Save CSV (read by thesis Rmd) ---
out_csv = OUTPUT_ROOT / 'dic_comparison.csv'
df.to_csv(out_csv, index=False)
print(f'Saved: {out_csv}')

In [ ]:
# --- Save xlsx (exploratory reference) ---
R_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
out_xlsx = R_OUTPUT_ROOT / 'dic_comparison.xlsx'
df.to_excel(out_xlsx, index=False)
print(f'Saved: {out_xlsx}')